<a href="https://colab.research.google.com/github/claudio1975/Medium-blog/blob/master/Fine-tuning_LLMS_on_medical_dataset_with_Unsloth/Qwen3_0_6B_fine_tuned_OpenMed_v2_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# STEP 1: INSTALL DEPENDENCIES
# ============================================================================


print("📦 Installing Unsloth and dependencies...")
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes
print("✅ Installation complete!")


In [ ]:
# ============================================================================
# STEP 2: IMPORT LIBRARIES
# ============================================================================

print("📚 Importing libraries...")
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments, TextStreamer
import pandas as pd
import os
import json

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")


📚 Importing libraries...
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ PyTorch version: 2.9.1+cu128
✅ CUDA available: True
✅ GPU: NVIDIA A100-SXM4-40GB
✅ GPU Memory: 39.56 GB


In [ ]:
# ============================================================================
# STEP 3: LOAD MODEL
# ============================================================================

print("\n⚙️ Setting up configuration...")

# Model settings
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3-0.6B",
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = False,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    dtype = torch.float16,
    full_finetuning = False,

)



In [ ]:
# ============================================================================
# STEP 4: LORA ADAPTERS
# ============================================================================


model = FastLanguageModel.get_peft_model(
    model,
    r = 32,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Trainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"✅ Total parameters: {total_params:,}")


In [ ]:
# ============================================================================
# STEP 5: LOAD AND INSPECT DATASET
# ============================================================================

print("\n📊 Loading dataset: OpenMed/Medical-Reasoning-SFT-GPT-OSS-120B")
print("⏳ This may take a moment...")

# Load dataset
dataset = load_dataset("OpenMed/Medical-Reasoning-SFT-GPT-OSS-120B", split="train")

print(f"✅ Dataset loaded! Total samples: {len(dataset):,}")
print(f"\n📋 Dataset columns: {dataset.column_names}")
print(f"\n🔍 First sample:")
print(dataset[0])


In [ ]:
# ============================================================================
# STEP 6: FORMAT DATASET
# ============================================================================

def format_conversation(example):
    conversation = ""
    for msg in example["messages"]:
        role = msg["role"]
        content = msg["content"]
        conversation += f"{role}: {content}\n"
    return {"text": conversation}

formatted_dataset = dataset.map(format_conversation)

In [ ]:
formatted_dataset[0]

{'messages': [{'content': 'Please summarize the following context:\n\nfor those with service-related injuries who have not applied for benefits: 1-800-487-7797.105\n\nRainbow Veterans is a Veterans group that offers support to members who experienced discrimination while in the CAF because of their sexual orientation.\n\n# Current military Veteran health concerns\n\nBelow are a few examples of health issues of concern to military Veterans of Canada from all eras:\n\n![](https://patientsmedicalhome.ca/files/uploads/images/cc6b69966907abcda71536b1e3a38f33fe4991443de2b2286ffdbf85f74b86ab.jpg)\n\ncan be treated using traditional approaches. In a comprehensive study on Gulf War illnesses, the National Academy of Medicine (formerly Institute of Medicine) in the United States recommends that providers employ a long-term, integrated approach to helping patients manage their symptoms.\n\n![](https://patientsmedicalhome.ca/files/uploads/images/4a885302c7dff065c9168d1b2af421a232187e35b1525dc3cab7

In [ ]:
data = pd.Series(formatted_dataset)
data.name = "text"


In [ ]:
# ============================================================================
# STEP 7: CONFIGURE TRAINER
# ============================================================================

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 8, # Use GA to mimic batch size!
        warmup_steps = 5,
        #num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 30,
        learning_rate = 2e-5, # Reduce to 2e-5 for long training runs
        logging_steps = 100,
        save_steps=500,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
# ============================================================================
# STEP 8: TRAIN!
# ============================================================================

print("\n" + "="*70)
print("🚀 STARTING TRAINING!")
print("="*70)
print("="*70 + "\n")

# Show GPU memory before training
if torch.cuda.is_available():
    print(f"💾 GPU Memory before training: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# Train!
trainer_stats = trainer.train()

print("\n" + "="*70)
print("🎉 TRAINING COMPLETED!")
print("="*70)
print(f"⏱️  Total training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"⚡ Samples/second: {trainer_stats.metrics['train_samples_per_second']:.2f}")
print(f"📉 Final loss: {trainer_stats.metrics['train_loss']:.4f}")
print("="*70 + "\n")



In [ ]:
messages = [
    {"role" : "user", "content" : "What are the main symptoms of pneumonia?"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = False, # Disable thinking
)

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 256, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Pneumonia is a lung infection that can be caused by bacteria, viruses, or fungi. The main symptoms vary depending on the type of pneumonia and the patient's age, but commonly include:

1. **Cough** – Often severe and persistent.
2. **Sore throat** – Especially if it's viral.
3. **Fatigue** – Due to the body's effort to fight the infection.
4. **Shortness of breath** – Especially if the infection is severe.
5. **High fever** – Especially in bacterial pneumonia.
6. **Chills or colds** – May be a sign of a bacterial infection.
7. **Malaise or weakness** – If the infection is severe or spreading.
8. **Fatigue** – Especially if the infection is spreading.
9. **Dyspnea** – Shortness of breath or difficulty breathing.
10. **Cough with blood or sputum** – If the infection is bacterial.

It's important to seek medical attention if symptoms are severe or persistent, as pneumonia can be serious and may require antibiotics or hospitalization. If you're experiencing symptoms like a persistent cough

In [ ]:
messages = [
    {"role" : "user", "content" : "What are the main symptoms of diabetes"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = False, # Disable thinking
)

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 256, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Diabetes is a metabolic disorder characterized by high blood sugar levels. The main symptoms include:

1. **Frequent urination**: Due to the kidneys working harder to filter out excess glucose.

2. **Increased thirst**: The body tries to excrete excess sugar through urine, leading to frequent thirst.

3. **Dark urine**: The urine appears dark because it has more glucose.

4. **Weight changes**: Although weight loss can be a sign of diabetes, weight gain may also occur, depending on the type of diabetes.

5. **Fatigue**: The body is working hard to process sugar, leading to fatigue.

6. **Blurred vision**: High blood sugar can cause the blood vessels to become swollen, leading to blurred vision.

7. **Nausea and vomiting**: These symptoms may occur in people with type 2 diabetes, especially after eating.

8. **Dry mouth**: The mouth may become dry because the body is using up glucose to process sugar.

9. **Fatigue**: The body is working hard to process sugar, leading to fatigue.

10. *

In [ ]:
messages = [
    {"role" : "user", "content" : "What are the main symptoms of heart disease?"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
    enable_thinking = False, # Disable thinking
)

_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 256, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Heart disease can have a variety of symptoms, depending on the type and severity of the condition. Here are some common symptoms:

1. **Heartburn**: Especially after eating, can be a symptom of heartburn or GERD.

2. **Shortness of breath**: Especially during exercise or when resting.

3. **Chest pain or pressure**: Often described as a "crackle" or "aching" sensation.

4. **Fatigue**: Especially after exertion.

5. **Dizziness or fainting**: Especially when standing up quickly.

6. **Nausea or vomiting**: Often associated with heartburn.

7. **Cold sweats**: Often associated with heartburn.

8. **Swelling in the legs or ankles**: Often associated with heart failure.

9. **Lightheadedness or dizziness**: Especially when standing up quickly.

10. **Fever**: Often associated with heartburn or infection.

11. **Aching or sore throat**: Often associated with heartburn.

12. **Numbness or tingling**: Often associated with heartburn.

13. **Cough or sore throat**: Often associated with heart

In [ ]:
# ============================================================================
# STEP 10: SAVE MODEL
# ============================================================================

print("💾 Saving model locally...")

# Save LoRA adapters
OUTPUT_DIR="qwen3-medical-output"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ Model saved to: {OUTPUT_DIR}")



In [ ]:

print("💾 Saving model on HF_HUB...")

model.push_to_hub("towardsinnovationlab/qwen3-medical", token = "") # Online saving
tokenizer.push_to_hub("towardsinnovationlab/qwen3-medical", token = "") # Online saving

print(f"✅ Model saved to: towardsinnovationlab/qwen3-medical")
